# SIFT

In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def perform_sift(image_path):

    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    if image is None:
        raise ValueError("Could not load image. Check the file path.")

    sift = cv2.SIFT_create()

    keypoints, descriptors = sift.detectAndCompute(image, None)

    sift_image = cv2.drawKeypoints(image, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    plt.figure(figsize=(8, 6))
    plt.imshow(sift_image, cmap="gray")
    plt.axis("off")  # Hide axis
    plt.title("SIFT Keypoints")
    plt.show()

    return keypoints, descriptors

keypoints, descriptors = perform_sift(r"D:\Sample-image-Lena-image-size-512-512-pixels-clustered-by-the-original-SLIC-middle.png")

# Image Classification

In [3]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from skimage.feature import hog

def extract_features(img, method='sift'):
    img = cv2.resize(img, (128, 128))  # Resize for SIFT/GLOH compatibility
    if method == 'sift':
        sift = cv2.SIFT_create()
        kp, desc = sift.detectAndCompute(img, None)
        if desc is None:
            desc = np.zeros((1, 128))
        return desc.flatten()

    elif method == 'hog':
        img = cv2.resize(img, (64, 64))  # Resize for HOG
        features = hog(img, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), visualize=False, multichannel=False)
        return features

    elif method == 'gloh':
        sift = cv2.SIFT_create()
        kp, desc = sift.detectAndCompute(img, None)
        if desc is None:
            desc = np.zeros((1, 128))
        return np.mean(desc, axis=0)

    else:
        raise ValueError("Unknown method")

digits = load_digits()
images = digits.images
labels = digits.target

images = ((images / images.max()) * 255).astype(np.uint8)

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.2, random_state=42)

method = 'sift'  

X_train_feats = [extract_features(img, method) for img in X_train]
X_test_feats = [extract_features(img, method) for img in X_test]

max_len = max(max(len(f) for f in X_train_feats), max(len(f) for f in X_test_feats))
X_train_feats = [np.pad(f, (0, max_len - len(f))) for f in X_train_feats]
X_test_feats = [np.pad(f, (0, max_len - len(f))) for f in X_test_feats]

clf = make_pipeline(StandardScaler(), SVC(kernel='linear'))
clf.fit(X_train_feats, y_train)

y_pred = clf.predict(X_test_feats)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.88      0.87        33
           1       0.76      0.93      0.84        28
           2       0.82      0.85      0.84        33
           3       0.69      0.79      0.74        34
           4       0.87      0.87      0.87        46
           5       0.72      0.77      0.74        47
           6       0.88      0.86      0.87        35
           7       0.89      0.71      0.79        34
           8       0.81      0.83      0.82        30
           9       0.81      0.62      0.70        40

    accuracy                           0.81       360
   macro avg       0.81      0.81      0.81       360
weighted avg       0.81      0.81      0.80       360

